# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [3]:
import requests  # Imports requests so Python can download each catalog page from the website.
import pandas as pd  # Imports pandas so the scraped book records can be stored in a table.
from bs4 import BeautifulSoup  # Imports BeautifulSoup so Python can read and search the HTML structure.
from urllib.parse import urljoin  # Imports urljoin so relative book/category links become full URLs.

RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}  # Maps Books to Scrape rating words to numeric scores.
BASE_URL = "https://books.toscrape.com/"  # Stores the website home page used as the starting point for scraping.
CATALOGUE_URL = urljoin(BASE_URL, "catalogue/")  # Stores the catalogue base URL used to follow pagination links.


def price_to_float(price_text):  # Defines a helper that converts price text such as "?12.99" into a number.
    cleaned_price = price_text.replace("?", "").strip()  # Removes the pound symbol and extra whitespace from the price.
    return float(cleaned_price)  # Converts the cleaned price string into a floating point number.


def scrape_books(min_rating=4, max_price=20):  # Defines the main scraping function requested by the lab.
    session = requests.Session()  # Creates a reusable HTTP session for the website requests.
    current_url = urljoin(CATALOGUE_URL, "page-1.html")  # Starts scraping from the first catalogue page.
    books = []  # Creates an empty list that will hold one dictionary per matching book.

    while current_url is not None:  # Keeps scraping until there is no next-page link left.
        response = session.get(current_url, timeout=20)  # Downloads the current catalogue page with a timeout.
        response.raise_for_status()  # Stops the notebook with a clear error if the page request failed.
        soup = BeautifulSoup(response.text, "html.parser")  # Parses the downloaded HTML into a searchable object.
        book_cards = soup.select("article.product_pod")  # Selects every book card shown on the current page.

        for card in book_cards:  # Loops through each book card on the page.
            title = card.select_one("h3 a")["title"]  # Reads the full book title from the title attribute.
            price = price_to_float(card.select_one(".price_color").get_text(strip=True))  # Converts the visible price to a number.
            rating_word = card.select_one("p.star-rating")["class"][1]  # Reads the word rating class such as Four or Five.
            rating = RATING_MAP[rating_word]  # Converts the word rating into a numeric rating.
            availability = card.select_one(".availability").get_text(" ", strip=True)  # Reads the availability text from the card.
            relative_link = card.select_one("h3 a")["href"]  # Reads the relative URL to the book details page.
            book_url = urljoin(current_url, relative_link)  # Converts the relative book URL into a full URL.

            if rating >= min_rating and price <= max_price:  # Keeps only books that match the rating and price filters.
                books.append({"title": title, "price": price, "rating": rating, "availability": availability, "url": book_url})  # Adds the matching book to the result list.

        next_link = soup.select_one("li.next a")  # Looks for the pagination link to the next catalogue page.
        current_url = urljoin(current_url, next_link["href"]) if next_link else None  # Moves to the next page or stops the loop.

    return pd.DataFrame(books, columns=["title", "price", "rating", "availability", "url"])  # Returns the final table of scraped books.


books_df = scrape_books(min_rating=4, max_price=20)  # Scrapes books rated 4 or higher with a price of 20 pounds or less.
print(f"Found {len(books_df)} matching books.")# Summarizes how many books matched the requested filters.
display(books_df.head(20))# Provides a compact notebook preview for review.


ValueError: could not convert string to float: 'Â£51.77'